In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#%matplotlib inline

#import seaborn as sns


import os, sys, shutil, importlib, glob
from tqdm.notebook import tqdm
# from celloracle import motif_analysis as ma
# import celloracle as co


In [2]:
import glob as glob

In [3]:
networks = glob.glob('/data2st1/junyi/output/atac1112/cicre/*network.csv')
df_peaks = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/cCRE_annotated.csv')

In [4]:
df_peaks.primary_region.value_counts()

primary_region
genebody      886180
downstream    367739
distal        308672
promoter       63154
intron         30464
exon            6628
UTR             4495
Name: count, dtype: int64

In [5]:
# peaks = df_peaks.names.str.replace("[:-]","_")
# tss_annotated = ma.get_tss_info(peak_str_list=peaks, ref_genome="mm10")


In [6]:
df_peaks_promoter = df_peaks[df_peaks['primary_region']=='promoter']
df_peaks_promoter['Peakp'] = df_peaks_promoter['names'].str.replace("[:-]","_")

/tmp/ipykernel_2371074/1831468524.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_peaks_promoter['Peakp'] = df_peaks_promoter['names'].str.replace("[:-]","_")


In [7]:
df_peaks_promoter

,chr,start,end,names,gene_name,gene_id,gstart,gend,strand,annotation_x,distance,primary_region,secondary_region,encodeCCRE,Sex,Ensemble,Gene,Peakp
20,chr1,3071402,3071903,chr1:3071402-3071903,4933401J01Rik,ENSMUSG00000102693.1,3073252,3074322,+,genebody,1350,promoter,NaN,NaN,M,ENSMUSG00000102693.1,chr1:3071402-3071903,chr1:3071402-3071903
21,chr1,3072947,3073448,chr1:3072947-3073448,4933401J01Rik,ENSMUSG00000102693.1,3073252,3074322,+,genebody,0,promoter,NaN,NaN,M,ENSMUSG00000102693.1,chr1:3072947-3073448,chr1:3072947-3073448
32,chr1,3100678,3101179,chr1:3100678-3101179,Gm26206,ENSMUSG00000064842.1,3102015,3102125,+,genebody,837,promoter,NaN,NaN,M,ENSMUSG00000064842.1,chr1:3100678-3101179,chr1:3100678-3101179
224,chr1,3368193,3368694,chr1:3368193-3368694,Gm37180,ENSMUSG00000103377.1,3365730,3368549,-,genebody,0,promoter,NaN,NaN,M,ENSMUSG00000103377.1,chr1:3368193-3368694,chr1:3368193-3368694
306,chr1,3466283,3466784,chr1:3466283-3466784,Gm1992,ENSMUSG00000089699.1,3466586,3513553,+,genebody,0,promoter,NaN,NaN,M,ENSMUSG00000089699.1,chr1:3466283-3466784,chr1:3466283-3466784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1667202,chrY,72822100,72822601,chrY:72822100-72822601,Gm20846,ENSMUSG00000100869.1,72819637,72820330,-,genebody,-1771,promoter,NaN,NaN,M,ENSMUSG00000100869.1,chrY:72822100-72822601,chrY:72822100-72822601
1667239,chrY,86219218,86219719,chrY:86219218-86219719,Gm28218,ENSMUSG00000099824.1,86219343,86222581,+,genebody,0,promoter,NaN,NaN,M,ENSMUSG00000099824.1,chrY:86219218-86219719,chrY:86219218-86219719
1667246,chrY,86870132,86870633,chrY:86870132-86870633,Gm21462,ENSMUSG00000100655.1,86869359,86870055,-,genebody,-78,promoter,NaN,NaN,M,ENSMUSG00000100655.1,chrY:86870132-86870633,chrY:86870132-86870633
1667262,chrY,88052253,88052754,chrY:88052253-88052754,Gm28102,ENSMUSG00000101915.1,88053314,88079494,+,genebody,561,promoter,NaN,NaN,M,ENSMUSG00000101915.1,chrY:88052253-88052754,chrY:88052253-88052754


In [8]:
enhancers = pd.DataFrame()
for network in networks:
    df_cire_tmp = pd.read_csv(network,index_col=0)
    df_cire_selected = df_cire_tmp[df_cire_tmp.pval_adj<0.05]
    #df_cire_selected = df_cire_selected.rename({"score":"coaccess"},axis=1)

    df_enh1 = df_cire_selected.merge(df_peaks_promoter[['Peakp','gene_name']],left_on='Peak1',right_on='Peakp',how='inner')[['Peak2','Peakp','score','gene_name']]
    df_enh1.columns = ['peak_id','promoter','coaccess','gene_short_name']
    df_enh2 = df_cire_selected.merge(df_peaks_promoter[['Peakp','gene_name']],left_on='Peak2',right_on='Peakp',how='inner')[['Peak1','Peakp','score','gene_name']]
    df_enh2.columns = ['peak_id','promoter','coaccess','gene_short_name']
    integrated_enhancer = pd.concat([df_enh1,df_enh2],axis=0)
    integrated_enhancer.sort_values(by='coaccess',ascending=False,inplace=True)
    integrated_enhancer.drop_duplicates(subset='peak_id', inplace=True)

    if df_cire_selected.shape[0]==0:
        continue
    # integrated = ma.integrate_tss_peak_with_cicero(tss_peak=tss_annotated,
    #                                             cicero_connections=df_cire_selected)
    # integrated_enhancer = integrated[integrated.coaccess<1]
    integrated_enhancer_split = integrated_enhancer.peak_id.str.split("_",expand=True)
    integrated_enhancer['names'] = integrated_enhancer_split[0] + ":" + integrated_enhancer_split[1].astype(str) + "-" + integrated_enhancer_split[2].astype(str)
    region_subclass = network.split('/')[-1].replace('_ALL_circe_network.csv','').replace('HIP','HPF').replace('AMY_AMY','AMY').replace('HPF_HPF','HPF').replace('PFC_PFC','PFC')
    integrated_enhancer['Region Subclass'] = region_subclass
    integrated_enhancer.to_csv(network.replace('_ALL_circe_network.csv','_enhancers.csv'))
    enhancers = pd.concat([enhancers,integrated_enhancer],axis=0)

KeyError: 0

In [9]:
fenhancers = glob.glob('/data2st1/junyi/output/atac1112/cicre/*enhancers.csv')
enhancers = pd.DataFrame()
for enhancer in fenhancers:
    df_tmp = pd.read_csv(enhancer,index_col=0)
    enhancers = pd.concat([enhancers,df_tmp],axis=0)

In [10]:
df_deg = pd.read_csv("/data2st1/junyi/output/atac1112/dar/celltype.L2/mast_ngsa_noco_degs_fdr_log2fc0_filtered.csv")
df_deg['Region Subclass'] = df_deg['Region subclass'].str.replace(" ",'_')
df_deg['Region Subclass'] = df_deg['Region Subclass'].str.replace("/",'-')
df_deg = df_deg[df_deg.Sex=='M']
df_deg = df_deg[df_deg.Region.isin(['HPF','PFC','AMY'])]

In [11]:
enhancers

,peak_id,promoter,coaccess,gene_short_name,names,Region Subclass
1,chr18_83185522_83186023,chr18_83399371_83399872,0.908645,Gm50413,chr18:83185522-83186023,AMY_MOL-1
0,chr1_171713668_171714169,chr1_171710750_171711251,0.853936,Tma7-ps,chr1:171713668-171714169,AMY_MOL-1
34,chr7_83857882_83858383,chr7_83872953_83873454,0.850116,Gm44530,chr7:83857882-83858383,AMY_MOL-1
38,chr2_19368695_19369196,chr2_19369259_19369760,0.841803,Msrb2,chr2:19368695-19369196,AMY_MOL-1
69,chr18_86921617_86922118,chr18_86927209_86927710,0.837851,Gm5971,chr18:86921617-86922118,AMY_MOL-1
...,...,...,...,...,...,...
26578,chr3_36202901_36203402,chr3_36000461_36000962,0.136311,Mccc1,chr3:36202901-36203402,HPF_Mossy_Glut
31923,chr13_67011640_67012141,chr13_67360387_67360888,0.136309,Gm17039,chr13:67011640-67012141,HPF_Mossy_Glut
32285,chr3_54457467_54457968,chr3_54155242_54155743,0.136307,Trpc4,chr3:54457467-54457968,HPF_Mossy_Glut
3867,chr8_122867660_122868161,chr8_123236030_123236531,0.136304,Gm45842,chr8:122867660-122868161,HPF_Mossy_Glut


In [ ]:
df_enhcer_deg = enhancers.merge(df_deg,left_on=['gene_short_name','Region Subclass'],right_on=['Gene','Region Subclass'],how='right')

In [12]:
df_DAR = pd.read_csv('/data2st1/junyi/output/atac1112/dar/celltype.L2/MASTNG_dar_annotated.csv')

In [13]:
df_DAR_promoter=df_DAR[df_DAR.primary_region.isin(['promoter'])]

In [15]:
df_DAR_promoter.to_csv('/data2st1/junyi/output/atac1112/dar/celltype.L2/MASTNG_dar_promoter.csv')

In [59]:
df_DAR_np = df_DAR[~df_DAR.primary_region.isin(['promoter'])]

In [107]:
df_DAR.primary_region.value_counts()

promoter      179665
intron         62220
downstream     27927
distal         21888
exon           19345
UTR            18810
genebody        4282
Name: primary_region, dtype: int64

In [ ]:
from sys import prefix


df_encher_deg_merged = df_enhcer_deg.merge(df_DAR_np,left_on=['names','Region Subclass','Direction'],right_on=['names','Region Subclass','Direction'],how='inner')

In [136]:
df_DAR_enh = enhancers.merge(df_DAR_np,left_on=['names','Region Subclass'],right_on=['names','Region Subclass'],how='inner')

In [138]:
df_DAR

,names,Pr(>Chisq),coef,ci.hi,ci.lo,padjust,celltype.L2,region,ctname,fdr,...,Subclass,Sex,Region Subclass,Ensemble,Gene,log2FC,Direction,Neurotransmitter,Gene_name,FDR
0,chr10:100456562-100457063,1.414898e-02,-0.006031,0.006165,-0.018226,1.000000,HPF_CA3_Glut,HPF,HPF_CA3_Glut,2.751694e-02,...,HPF_CA3_Glut,M,HPF_CA3_Glut,ENSMUSG00000036676.14,chr10:100456562-100457063,-0.006031,Down,Glut,chr10:100456562-100457063,2.751694e-02
1,chr10:100487209-100487710,1.702931e-10,0.068210,0.093457,0.042963,0.000006,HPF_CA3_Glut,HPF,HPF_CA3_Glut,9.867574e-08,...,HPF_CA3_Glut,M,HPF_CA3_Glut,ENSMUSG00000036676.14,chr10:100487209-100487710,0.068210,Up,Glut,chr10:100487209-100487710,9.867574e-08
2,chr10:100487209-100487710,1.702931e-10,0.068210,0.093457,0.042963,0.000006,HPF_CA3_Glut,HPF,HPF_CA3_Glut,9.867574e-08,...,HPF_CA3_Glut,M,HPF_CA3_Glut,ENSMUSG00000036676.14,chr10:100487209-100487710,0.068210,Up,Glut,chr10:100487209-100487710,9.867574e-08
3,chr10:100642797-100643298,6.163128e-03,-0.014478,-0.004093,-0.024864,1.000000,HPF_CA3_Glut,HPF,HPF_CA3_Glut,1.490774e-02,...,HPF_CA3_Glut,M,HPF_CA3_Glut,ENSMUSG00000112328.1,chr10:100642797-100643298,-0.014478,Down,Glut,chr10:100642797-100643298,1.490774e-02
4,chr10:10107158-10107659,5.303556e-07,0.030881,0.045954,0.015808,0.017210,HPF_CA3_Glut,HPF,HPF_CA3_Glut,2.347109e-05,...,HPF_CA3_Glut,M,HPF_CA3_Glut,ENSMUSG00000112713.1,chr10:10107158-10107659,0.030881,Up,Glut,chr10:10107158-10107659,2.347109e-05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334132,chr7:99535252-99535753,4.700413e-06,0.001785,0.060391,-0.056820,0.079099,AMY_Npas1_Rgs12_GABA,AMY,AMY_Npas1_Rgs12_GABA,1.977464e-02,...,AMY_Npas1_Rgs12_GABA,M,AMY_Npas1_Rgs12_GABA,ENSMUSG00000018909.15,chr7:99535252-99535753,0.001785,Up,GABA,chr7:99535252-99535753,1.977464e-02
334133,chr9:103365644-103366145,8.286317e-05,0.028069,0.108636,-0.052498,1.000000,AMY_Npas1_Rgs12_GABA,AMY,AMY_Npas1_Rgs12_GABA,4.904168e-02,...,AMY_Npas1_Rgs12_GABA,M,AMY_Npas1_Rgs12_GABA,ENSMUSG00000032803.15,chr9:103365644-103366145,0.028069,Up,GABA,chr9:103365644-103366145,4.904168e-02
334134,chr9:99436435-99436936,3.917479e-05,-0.054067,-0.014330,-0.093805,0.659233,AMY_Npas1_Rgs12_GABA,AMY,AMY_Npas1_Rgs12_GABA,3.488255e-02,...,AMY_Npas1_Rgs12_GABA,M,AMY_Npas1_Rgs12_GABA,ENSMUSG00000032470.17,chr9:99436435-99436936,-0.054067,Down,GABA,chr9:99436435-99436936,3.488255e-02
334135,chr13:43171217-43171718,2.009124e-06,NaN,NaN,NaN,0.018430,PFC_Sst_Chodl_GABA,PFC,PFC_Sst_Chodl_GABA,1.842970e-02,...,PFC_Sst_Chodl_GABA,M,PFC_Sst_Chodl_GABA,ENSMUSG00000021368.15,chr13:43171217-43171718,NaN,Down,GABA,chr13:43171217-43171718,1.842970e-02


In [127]:
df_promoter_deg_merged = df_deg.merge(df_DAR_promoter,left_on=['Gene','Region Subclass','Direction'],right_on=['gene_name','Region Subclass','Direction'],how='left')   

In [128]:
df_promoter_deg_merged

,Gene_x,pval,log2FC_x,ci.hi_x,ci.lo_x,FDR_x,Bonferroni,Subclass_x,Region_x,Sex_x,...,encodeCCRE,Region_y,Subclass_y,Sex_y,Ensemble_y,Gene_y,log2FC_y,Neurotransmitter_y,Gene_name_y,FDR_y
0,1810062O18Rik,5.808074e-03,0.070354,0.120168,0.020540,0.040524,1.000000,Astrocyte-2,PFC,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2610035D17Rik,7.490775e-04,0.088500,0.139018,0.037981,0.009548,1.000000,Astrocyte-2,PFC,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4732471J01Rik,8.759375e-04,0.053831,0.097587,0.010075,0.010621,1.000000,Astrocyte-2,PFC,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4833420G17Rik,4.983856e-03,0.040075,0.087102,-0.006953,0.036435,1.000000,Astrocyte-2,PFC,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4930402H24Rik,7.804847e-04,0.105143,0.164551,0.045736,0.009823,1.000000,Astrocyte-2,PFC,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80669,Zswim6,4.582351e-03,-0.098474,-0.027546,-0.169402,0.039133,1.000000,OPC,AMY,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80670,Apoe,4.143067e-07,0.876975,1.205846,0.548104,0.000844,0.001689,VLMC,AMY,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80671,Meg3,3.685718e-07,0.896927,1.220253,0.573601,0.000844,0.001502,VLMC,AMY,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80672,Slc6a13,3.642790e-06,0.890118,1.255269,0.524966,0.004949,0.014848,VLMC,AMY,M,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [131]:
#write cross on Direction_y AND Direction_x 
df_crosstab = pd.crosstab(df_DAR_enh['Direction_x'], df_DAR_enh['Direction_y'])

In [132]:
df_crosstab

Direction_y,Down,Up
Direction_x,,
Down,151,149
Up,137,195


In [76]:
df_promoter_deg_merged.corr()

/tmp/ipykernel_2725342/857190339.py:1: FutureWarning: The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  df_promoter_deg_merged.corr()


,pval,log2FC_x,ci.hi_x,ci.lo_x,FDR_x,Bonferroni,Pr(>Chisq),coef,ci.hi_y,ci.lo_y,padjust,fdr,gstart,gend,distance,log2FC_y,FDR_y
pval,1.000000,-0.029112,-0.058810,0.002183,0.823400,0.475565,-0.001201,0.042815,0.041531,0.037868,-0.017586,-0.018948,0.002499,0.002115,-0.002890,0.042815,-0.018948
log2FC_x,-0.029112,1.000000,0.972361,0.972382,-0.031283,-0.049555,-0.028780,0.022625,0.054591,-0.023166,-0.045869,-0.026666,0.037438,0.035778,-0.020699,0.022625,-0.026666
ci.hi_x,-0.058810,0.972361,1.000000,0.891013,-0.016543,-0.055678,-0.028964,0.016692,0.073984,-0.061676,-0.040369,-0.020884,0.034011,0.032496,-0.020093,0.016692,-0.020884
ci.lo_x,0.002183,0.972382,0.891013,1.000000,-0.044290,-0.040696,-0.027963,0.028044,0.034048,0.015751,-0.050343,-0.031846,0.040032,0.038264,-0.020848,0.028044,-0.031846
FDR_x,0.823400,-0.031283,-0.016543,-0.044290,1.000000,0.602009,0.007609,0.026711,0.051032,-0.009603,0.014495,0.022490,0.004233,0.003838,-0.003408,0.026711,0.022490
Bonferroni,0.475565,-0.049555,-0.055678,-0.040696,0.602009,1.000000,0.007770,0.054352,0.076521,0.016596,0.006878,0.013750,0.006276,0.005610,0.000424,0.054352,0.013750
Pr(>Chisq),-0.001201,-0.028780,-0.028964,-0.027963,0.007609,0.007770,1.000000,-0.156196,-0.188175,-0.089661,0.362867,0.908887,-0.007382,-0.007162,-0.012474,-0.156196,0.908887
coef,0.042815,0.022625,0.016692,0.028044,0.026711,0.054352,-0.156196,1.000000,0.950069,0.910857,-0.316985,-0.159289,-0.012055,-0.012415,0.008910,1.000000,-0.159289
ci.hi_y,0.041531,0.054591,0.073984,0.034048,0.051032,0.076521,-0.188175,0.950069,1.000000,0.736591,-0.330270,-0.159490,-0.011334,-0.011791,0.006125,0.950069,-0.159490
ci.lo_y,0.037868,-0.023166,-0.061676,0.015751,-0.009603,0.016596,-0.089661,0.910857,0.736591,1.000000,-0.250223,-0.134304,-0.011137,-0.011313,0.011211,0.910857,-0.134304


In [63]:
df_encher_deg_merged.primary_region.value_counts()

exon          254
intron        115
UTR            95
downstream     78
distal         76
genebody       14
Name: primary_region, dtype: int64

In [3]:
df_tobias = pd.read_parquet('/data2st1/junyi/output/atac1112/tobias/annotated_filtered.parquet')

In [6]:
set(enhancers['Region Subclass']).difference(set(df_tobias['Region Subclass']))

{'AMY_Arachnoid_Barrier_cell',
 'AMY_Ependymal_cell',
 'AMY_MOL-3',
 'AMY_NFOL',
 'AMY_Perivascular_Macrophage',
 'AMY_VLMC',
 'HPF_Astrocyte-3',
 'HPF_COP',
 'HPF_Ependymal_cell',
 'HPF_Immature_cell',
 'HPF_Subiculum_CT_Glut',
 'HPF_VLMC',
 'PFC_Astrocyte-3',
 'PFC_COP',
 'PFC_MOL-3',
 'PFC_Perivascular_Macrophage',
 'PFC_Pvalb_Vipr2_GABA',
 'PFC_VLMC'}

In [7]:
set(df_tobias['Region Subclass']).difference(set(enhancers['Region Subclass']))


{'HPF_OPC'}

In [8]:
df_tobias_enhancers = pd.merge(df_tobias,enhancers,on=['names','Region Subclass'],how='left')

In [9]:
df_tobias_enhancers.head()

,TFBS_chr,TFBS_start,TFBS_end,TFBS_name,TFBS_score,TFBS_strand,peak_chr,peak_start,peak_end,MC_score,...,primary_region,secondary_region,encodeCCRE,Sex,Ensemble,Gene,peak_id,promoter,coaccess,gene_short_name
0,chr1,3309945,3309954,metacluster_137.2cisbp__M01001_None,9.25971,-,chr1,3309937,3310438,0.11586,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN
1,chr1,3309945,3309954,metacluster_137.2cisbp__M01001_None,9.34861,-,chr1,3309937,3310438,0.36816,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN
2,chr1,3309945,3309954,metacluster_137.2cisbp__M01001_None,9.26355,-,chr1,3309937,3310438,0.46053,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN
3,chr1,3310134,3310143,metacluster_68.2cisbp__M00407_None,7.61715,+,chr1,3309937,3310438,0.20840,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN
4,chr1,3309944,3309955,metacluster_137.2transfac_public__M00173_None,7.82886,-,chr1,3309937,3310438,0.11586,...,intron,repeat,None,M,ENSMUSG00000051951.5,chr1:3309937-3310438,NaN,NaN,NaN,NaN


In [10]:
df_tobias_enhancers_candidate = df_tobias_enhancers[df_tobias_enhancers.coaccess.notnull()]

In [11]:
import re
import numpy as np
import pandas as pd

_interval_re = re.compile(r"^(chr[^_]+)_(\d+)_(\d+)$")

def _parse_interval_str(s: str):
    """Parse 'chr1_3309937_3310438' -> (chrom, start, end); invalid -> (None, None, None)."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return (None, None, None)
    m = _interval_re.match(str(s).strip())
    if not m:
        return (None, None, None)
    chrom = m.group(1)
    start = int(m.group(2))
    end = int(m.group(3))
    if start > end:
        start, end = end, start
    return (chrom, start, end)

def add_pedistance(df: pd.DataFrame, peak_col: str = "peak_id", promoter_col: str = "promoter") -> pd.DataFrame:
    """
    In-place add column 'pedistance' = nearest boundary gap between df[peak_col] and df[promoter_col].

    Rules:
      - Different chromosome -> inf
      - Overlap or touch -> 0
      - Else -> start_right - end_left
      - Invalid interval string -> NaN
    """
    peak_parsed = df[peak_col].map(_parse_interval_str)
    prom_parsed = df[promoter_col].map(_parse_interval_str)

    peak_chr = peak_parsed.map(lambda x: x[0])
    peak_start = peak_parsed.map(lambda x: x[1])
    peak_end = peak_parsed.map(lambda x: x[2])

    prom_chr = prom_parsed.map(lambda x: x[0])
    prom_start = prom_parsed.map(lambda x: x[1])
    prom_end = prom_parsed.map(lambda x: x[2])

    pedist = pd.Series(np.nan, index=df.index, dtype="float64")

    valid = peak_chr.notna() & prom_chr.notna()

    # Different chromosomes -> inf
    pedist.loc[valid & (peak_chr != prom_chr)] = np.inf

    # Same chromosome -> compute nearest boundary gap
    same = valid & (peak_chr == prom_chr)
    ps = peak_start[same].to_numpy()
    pe = peak_end[same].to_numpy()
    rs = prom_start[same].to_numpy()
    re_ = prom_end[same].to_numpy()

    # Ensure left interval is the one with smaller start
    left_end = np.where(ps <= rs, pe, re_)
    right_start = np.where(ps <= rs, rs, ps)

    pedist.loc[same] = np.where(right_start <= left_end, 0.0, (right_start - left_end).astype(float))

    df["pedistance"] = pedist
    return df


In [12]:
df_tobias_enhancers_candidate = add_pedistance(df_tobias_enhancers_candidate, peak_col='peak_id', promoter_col='promoter')

/tmp/ipykernel_3635406/718569337.py:62: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["pedistance"] = pedist


In [ ]:
promoter_prox = df_tobias_enhancers_candidate[(df_tobias_enhancers_candidate.primary_region!='promoter')&(df_tobias_enhancers_candidate.pedistance<=200)]
promoter_prox['PEstatus'] = 'promoter_proximal'
near_prox = df_tobias_enhancers_candidate[(df_tobias_enhancers_candidate.primary_region.isin(['distal','intron']))&\
                                          (df_tobias_enhancers_candidate.pedistance>200)&(df_tobias_enhancers_candidate.pedistance<=2000)]
near_prox['PEstatus'] = 'near_proximal_enhancer'
distal = df_tobias_enhancers_candidate[(df_tobias_enhancers_candidate.primary_region.isin(['distal','intron']))&(df_tobias_enhancers_candidate.pedistance>2000)&(df_tobias_enhancers_candidate.pedistance<=500_000)]
distal['PEstatus'] = 'distal_enhancer'
addcolumns = ['promoter','coaccess','gene_short_name','pedistance','PEstatus']
for addcol in addcolumns:
    df_tobias[addcol] = np.nan
df_tobias[df_tobias.primary_region=='promoter']['PEstatus'] = 'promoter'
for df, name in zip([promoter_prox, near_prox, distal],
                    ['promoter_proximal_enhancers','near_promoter_enhancers','distal_enhancers']):
    print(f"Saving {name} with shape {df.shape}")

    # 确保按 index 对齐写回
    df_tobias.loc[df.index, addcolumns] = df.loc[df.index, addcolumns]

/tmp/ipykernel_3635406/3927001270.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  promoter_prox['PEstatus'] = 'promoter_proximal'


In [19]:
df_tobias.to_parquet('/data2st1/junyi/output/atac1112/tobias/annotated_filtered_with_enhancers.parquet')

In [166]:
df_tobias_raw = pd.read_parquet('/data2st1/junyi/output/atac1112/tobias/annotated_filtered.parquet')